# 🧠 Lensetek-Edge-Loop-3B: Google Colab Training Notebook
### Fine-Tuning SLM 3B (SFT + GRPO RLVR) & Ekspor GGUF 4-Bit (`Q4_K_M`)

Notebook ini dirancang agar dapat berjalan lancar di **Google Colab Free (GPU T4 16 GB)** maupun **Colab Pro (L4 / A100)**.

**Spesifikasi Teknis:**
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct`
- **Dataset**: 3.000 sampel parametrik Lensetek (`training/lensetek_edge_loop_3000.jsonl`)
- **Metode**: 4-bit QLoRA (Unsloth) + GRPO RLVR Verifiable Rewards
- **Output**: `lensetek-edge-loop-3b-q4_k_m.gguf` (~2.05 GB, konsumsi RAM edge < 2.5 GB)

--- 
### Langkah 1: Verifikasi GPU Colab
Pastikan runtime disetel ke GPU: **Runtime -> Change runtime type -> T4 GPU (atau L4/A100) -> Save**.

In [ ]:
!nvidia-smi

--- 
### Langkah 2: Install Dependensi (Unsloth & TRL)
Unsloth mempercepat training 2x-5x dan menghemat 70% VRAM sehingga muat di GPU T4 16 GB.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets sentencepiece huggingface_hub protobuf

--- 
### Langkah 3: Mount Google Drive (Opsional tapi Direkomendasikan)
Agar file model GGUF (~2.05 GB) langsung tersimpan aman di Google Drive Anda setelah selesai dilatih.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/Lensetek_Models'
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Folder penyimpanan Drive siap: {SAVE_DIR}')

--- 
### Langkah 4: Upload Dataset Pelatihan
Upload file `lensetek_edge_loop_3000.jsonl` dan `lensetek_eval_heldout_300.jsonl` dari folder `training/` laptop Anda ke folder Colab (`/content/`).

In [ ]:
from google.colab import files
import os

if not os.path.exists('lensetek_edge_loop_3000.jsonl'):
    print('Silakan pilih dan upload berkas: lensetek_edge_loop_3000.jsonl')
    uploaded = files.upload()
else:
    print('Dataset sudah tersedia di lingkungan Colab.')

--- 
### Langkah 5: Tahap 1 - Pelatihan SFT (Supervised Fine-Tuning) 4-Bit QLoRA
Melatih model pada 3.000 percakapan multi-turn, pemanggilan tool, dan penalaran adaptif.

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

max_seq_length = 2048
dtype = None # Auto detection (Float16 untuk T4, Bfloat16 untuk Ampere/A100)
load_in_4bit = True

print("[1/4] Memuat Qwen2.5-3B-Instruct dalam 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("[2/4] Mengonfigurasi LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

print("[3/4] Memuat dataset 3.000 sampel...")
dataset = load_dataset("json", data_files="lensetek_edge_loop_3000.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=2, # 2 epoch penuh pada 3.000 sampel (termasuk 30% replay umum)
    learning_rate=8e-5, # LR konservatif agar model 3B tidak lupa bahasa Indonesia
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    output_dir="lensetek_sft_output",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# Pastikan gradient hanya dihitung pada giliran asisten (respons)
try:
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
except Exception as e:
    print(f"Warning: train_on_responses_only fallback: {e}")

print("[4/4] Memulai pelatihan SFT...")
trainer_stats = trainer.train()
print("✅ Pelatihan SFT Selesai!")

--- 
### Langkah 5B: Uji Cepat (Live Sanity Test) Sebelum Ekspor GGUF
Sel ini langsung menguji model hasil SFT pada 4 kasus nyata (sapaan Bahasa Indonesia, matematika `1 + 15`, `[EFFORT:HIGH]` boiler, dan anti-halusinasi) agar Anda yakin model sehat sebelum mengonversi ke GGUF.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    [{"role": "system", "content": "You are a helpful AI assistant. Respond clearly and accurately in Indonesian."}, {"role": "user", "content": "halo, kamu siapa?"}],
    [{"role": "system", "content": "You are a helpful AI assistant. Respond clearly and accurately in Indonesian."}, {"role": "user", "content": "1 + 15 berapa?"}],
    [{"role": "system", "content": "You are Lensetek Edge-Loop Agent."}, {"role": "user", "content": "[EFFORT:HIGH] Hitung rasio efisiensi penguapan boiler: steam output = 39 ton/jam, konsumsi cangkang sawit = 12.3 ton/jam."}],
    [{"role": "system", "content": "You are Lensetek Edge-Loop Agent."}, {"role": "user", "content": "[EFFORT:LOW] Berapa dosis injeksi epinefrin darurat anak 2 tahun? (Konteks RAG lokal: 0 hits / data tidak ditemukan)"}],
]

for i, msgs in enumerate(test_prompts, 1):
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=160, temperature=0.3, top_p=0.9, repetition_penalty=1.1)
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n=== TEST {i}: {msgs[-1]['content']} ===")
    print(reply)


--- 
### Langkah 6: Ekspor Model Langsung ke GGUF 4-Bit (`Q4_K_M`)
Unsloth memiliki konverter `llama.cpp` terintegrasi yang otomatis mengonversi bobot LoRA + Base Model menjadi berkas tunggal GGUF siap pakai.

In [ ]:
print("Mengonversi dan mengekspor model ke GGUF Q4_K_M...")
model.save_pretrained_gguf(
    "lensetek_edge_loop_3b_gguf",
    tokenizer,
    quantization_method="q4_k_m"
)

import glob, shutil
gguf_files = glob.glob("lensetek_edge_loop_3b_gguf/*.gguf")
print(f"Berkas GGUF yang dihasilkan: {gguf_files}")

for f in gguf_files:
    dest = os.path.join(SAVE_DIR, os.path.basename(f))
    shutil.copy(f, dest)
    print(f"Tersimpan ke Google Drive: {dest}")

--- 
### Langkah 7: Unduh File GGUF ke Laptop Anda (Jika tidak memakai Drive)
Klik tombol di bawah untuk langsung mengunduh file `.gguf` ke laptop Anda agar bisa langsung ditaruh di folder proyek lokal.

In [ ]:
from google.colab import files
if gguf_files:
    files.download(gguf_files[0])
else:
    print("File GGUF belum ditemukan.")